# 9 · Surapprentissage et sous-apprentissage — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch09_overfitting/03_notebook.ipynb)

La partie 0 vérifie tes réponses courtes aux quiz, aux rappels et aux exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou les tests de ta librairie `mylearn`). « Exécuter tout » va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « dans tes notes » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/ch09_overfitting/03_notebook.ipynb`, créée par `python tools/start_chapter.py 9`) : ce fichier-ci est mis à jour par Claude.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 9.Q1 à 9.Q10, 9.R2, 9.R3, 9.1 à 9.7, 9.9 | vérifier tes réponses courtes | 🧠 🔁 ✏️ ∂ 📈 | ★ à ★★★ | |
| A | 9.12 à 9.17 | sous- et surapprentissage, mesures d'erreur, features polynomiales, moindres carrés et Ridge dans mylearn.linear (*prochaine session de génération*) | | | |
| B | 9.18 à 9.23 | courbes de validation, chemins de régularisation, early stopping, courbes d'apprentissage, Ridge contre Lasso, Lasso par descente de coordonnées (*prochaine session de génération*) | | | |
| C | 9.24 à 9.27 | biais et variance mesurés par simulation, droites bayésiennes sur une grille (*prochaine session de génération*) | | | |
| D | 9.28 à 9.31 | bugs de régularisation, refactorisation testée, double descente, défi California (*prochaine session de génération*) | | | |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        if pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="9")  # your own library

## Objectifs et rappel express

- Diagnostiquer l'underfitting et l'overfitting sur des courbes d'entraînement, de validation et d'apprentissage.
- Écrire et vérifier contre scikit-learn les mesures d'erreur d'une régression, les features polynomiales, les moindres carrés, Ridge et Lasso.
- Appliquer l'early stopping avec patience, et choisir la régularisation par validation croisée.
- Mesurer le biais² et la variance d'une famille de modèles, et ajuster une droite à la manière bayésienne.

**Rappel express.** $\mathrm{MSE} = \frac{1}{n}\sum_i (y_i - \hat{y}_i)^2$, $\mathrm{MAE} = \frac{1}{n}\sum_i |y_i - \hat{y}_i|$, $R^2 = 1 - SS_{\text{res}}/SS_{\text{tot}}$ ; moindres carrés : $a = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sum (x_i - \bar{x})^2}$, $b = \bar{y} - a\bar{x}$ ; Ridge : $(\mathbf{X}_c^\top\mathbf{X}_c + \alpha\mathbf{I})\mathbf{w} = \mathbf{X}_c^\top\mathbf{y}_c$ ; Lasso : $w_j = S(\rho_j, \alpha)/z_j$ avec $S(z, \gamma) = \operatorname{signe}(z)\max(|z| - \gamma, 0)$ ; erreur attendue = biais² + variance + $\sigma^2$. En Python : `np.linalg.lstsq`, `np.linalg.solve`, `sklearn.preprocessing.PolynomialFeatures`, `sklearn.linear_model.LinearRegression`, `Ridge`, `Lasso`, `sklearn.model_selection.learning_curve`, `validation_curve`.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ 9.1, 9.4, 9.5 et 9.7, ∂ 9.2, 9.3 et 9.6, 📈 9.9)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"E"`) ; plusieurs choix : les lettres collées (`"BD"`) ; plusieurs nombres : une liste (`[4, 7]`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », le quiz Q11, le rappel R1, la réflexion et l'entretien se corrigent avec `05_solutions.md`.

**9.Q1 — Sur- ou sous-apprentissage ? Définitions et symptômes**

In [ ]:
# 9.Q1: the values you found on paper
answer_9_Q1a = ...  # a) the letter of your choice, e.g. "E"
answer_9_Q1b = ...  # b) the letter of your choice
answer_9_Q1c = ...  # c) the letter of your choice
answer_9_Q1d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_9_Q1a, answer_9_Q1b, answer_9_Q1c, answer_9_Q1d]):
    wb.check(f"9.Q1{letter}", answer)

**9.Q2 — Walter et sa moustache : qu'est-ce qui a été mal appris ?**

In [ ]:
# 9.Q2: the values you found on paper
answer_9_Q2a = ...  # a) the letter of your choice
answer_9_Q2b = ...  # b) the letter of your choice
answer_9_Q2c = ...  # c) True or False
answer_9_Q2d = ...  # d) the letter of your choice

for letter, answer in zip("abcd", [answer_9_Q2a, answer_9_Q2b, answer_9_Q2c, answer_9_Q2d]):
    wb.check(f"9.Q2{letter}", answer)

**9.Q3 — Sous-apprentissage : les vrais remèdes**

In [ ]:
# 9.Q3: the values you found on paper
answer_9_Q3a = ...  # a) the letter of your choice
answer_9_Q3b = ...  # b) the letters of the useful remedies, in alphabetical order, e.g. "BD"
answer_9_Q3c = ...  # c) True or False

for letter, answer in zip("abc", [answer_9_Q3a, answer_9_Q3b, answer_9_Q3c]):
    wb.check(f"9.Q3{letter}", answer)

**9.Q4 — Courbes d'erreur : où commence le surapprentissage ?**

In [ ]:
# 9.Q4: the values you found on paper
answer_9_Q4a = ...  # a) a whole number (an epoch of the table)
answer_9_Q4b = ...  # b) True or False
answer_9_Q4c = ...  # c) the letter of your choice
answer_9_Q4d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_9_Q4a, answer_9_Q4b, answer_9_Q4c, answer_9_Q4d]):
    wb.check(f"9.Q4{letter}", answer)

**9.Q5 — Un point isolé : frontière tordue ou frontière simple ?**

In [ ]:
# 9.Q5: the values you found on paper
answer_9_Q5a = ...  # a) the letter of your choice
answer_9_Q5b = ...  # b) True or False
answer_9_Q5c = ...  # c) the letter of your choice
answer_9_Q5d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_9_Q5a, answer_9_Q5b, answer_9_Q5c, answer_9_Q5d]):
    wb.check(f"9.Q5{letter}", answer)

**9.Q6 — Early stopping : quand s'arrêter, et pourquoi c'est délicat**

In [ ]:
# 9.Q6: the values you found on paper
answer_9_Q6a = ...  # a) the letter of your choice
answer_9_Q6b = ...  # b) the letter of your choice
answer_9_Q6c = ...  # c) a whole number
answer_9_Q6d = ...  # d) the letter of your choice

for letter, answer in zip("abcd", [answer_9_Q6a, answer_9_Q6b, answer_9_Q6c, answer_9_Q6d]):
    wb.check(f"9.Q6{letter}", answer)

**9.Q7 — Régularisation : ce que change λ**

In [ ]:
# 9.Q7: the values you found on paper
answer_9_Q7a = ...  # a) the letter of your choice
answer_9_Q7b = ...  # b) the letter of your choice
answer_9_Q7c = ...  # c) True or False
answer_9_Q7d = ...  # d) the letter of your choice
answer_9_Q7e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_9_Q7a, answer_9_Q7b, answer_9_Q7c, answer_9_Q7d, answer_9_Q7e]):
    wb.check(f"9.Q7{letter}", answer)

**9.Q8 — Pénalité sur les poids, dropout, batchnorm : même objectif ?**

In [ ]:
# 9.Q8: the values you found on paper
answer_9_Q8a = ...  # a) the letters, in alphabetical order, e.g. "BD"
answer_9_Q8b = ...  # b) True or False
answer_9_Q8c = ...  # c) the letter of your choice
answer_9_Q8d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_9_Q8a, answer_9_Q8b, answer_9_Q8c, answer_9_Q8d]):
    wb.check(f"9.Q8{letter}", answer)

**9.Q9 — Biais et variance : des propriétés d'une famille de courbes**

In [ ]:
# 9.Q9: the values you found on paper
answer_9_Q9a = ...  # a) True or False
answer_9_Q9b = ...  # b) the letter of your choice
answer_9_Q9c = ...  # c) the letter of your choice
answer_9_Q9d = ...  # d) the letter of your choice

for letter, answer in zip("abcd", [answer_9_Q9a, answer_9_Q9b, answer_9_Q9c, answer_9_Q9d]):
    wb.check(f"9.Q9{letter}", answer)

**9.Q10 — Courbes raides ou souples : qui a quel biais, quelle variance ?**

In [ ]:
# 9.Q10: the values you found on paper
answer_9_Q10a = ...  # a) one letter, "A" or "B"
answer_9_Q10b = ...  # b) one letter, "A" or "B"
answer_9_Q10c = ...  # c) True or False
answer_9_Q10d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_9_Q10a, answer_9_Q10b, answer_9_Q10c, answer_9_Q10d]):
    wb.check(f"9.Q10{letter}", answer)

**9.R2 — Ch. 6 : ce que mesure une cross-entropy utilisée comme loss**

In [ ]:
# 9.R2: the values you found on paper
answer_9_R2a = ...  # a) a number (3 decimals)
answer_9_R2b = ...  # b) a number (3 decimals)
answer_9_R2c = ...  # c) a whole number (1, 2, 3 or 4)
answer_9_R2d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_9_R2a, answer_9_R2b, answer_9_R2c, answer_9_R2d]):
    wb.check(f"9.R2{letter}", answer)

**9.R3 — Ch. 2 : biais et variance d'un estimateur, et le bootstrap**

In [ ]:
# 9.R3: the values you found on paper
answer_9_R3a = ...  # a) a number
answer_9_R3b = ...  # b) a number (3 decimals)
answer_9_R3c = ...  # c) a negative number
answer_9_R3d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_9_R3a, answer_9_R3b, answer_9_R3c, answer_9_R3d]):
    wb.check(f"9.R3{letter}", answer)

**Ex 9.1 — MSE et R² à la main sur cinq points**

In [ ]:
# Ex 9.1: the values you found on paper
answer_9_1a = ...  # a) a number
answer_9_1b = ...  # b) a number (3 decimals)
answer_9_1c = ...  # c) a number
answer_9_1d = ...  # d) a number (3 decimals)
answer_9_1e = ...  # e) a number (3 decimals)
answer_9_1f = ...  # f) a number
answer_9_1g = ...  # g) a number

for letter, answer in zip("abcdefg", [answer_9_1a, answer_9_1b, answer_9_1c, answer_9_1d, answer_9_1e, answer_9_1f, answer_9_1g]):
    wb.check(f"9.1{letter}", answer)

**Ex 9.2 — Moindres carrés : la meilleure droite par dérivées partielles**

In [ ]:
# Ex 9.2: the values you found on paper
answer_9_2a = ...  # a) a number
answer_9_2b = ...  # b) a number
answer_9_2c = ...  # c) a number
answer_9_2d = ...  # d) a number
answer_9_2e = ...  # e) a number
answer_9_2f = ...  # f) a whole number

for letter, answer in zip("abcdef", [answer_9_2a, answer_9_2b, answer_9_2c, answer_9_2d, answer_9_2e, answer_9_2f]):
    wb.check(f"9.2{letter}", answer)

**Ex 9.3 — Ridge en dimension 1 : w* = Σxy / (Σx² + λ)**

In [ ]:
# Ex 9.3: the values you found on paper
answer_9_3a = ...  # a) a number
answer_9_3b = ...  # b) a number
answer_9_3c = ...  # c) a number (4 decimals)
answer_9_3d = ...  # d) a number
answer_9_3e = ...  # e) a number (3 decimals)
answer_9_3f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_9_3a, answer_9_3b, answer_9_3c, answer_9_3d, answer_9_3e, answer_9_3f]):
    wb.check(f"9.3{letter}", answer)

**Ex 9.4 — Biais² et variance à partir d'un tableau de prédictions**

In [ ]:
# Ex 9.4: the values you found on paper
answer_9_4a = ...  # a) a list of three numbers
answer_9_4b = ...  # b) a number (3 decimals)
answer_9_4c = ...  # c) a number (3 decimals)
answer_9_4d = ...  # d) a number (3 decimals)
answer_9_4e = ...  # e) a number (3 decimals)
answer_9_4f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_9_4a, answer_9_4b, answer_9_4c, answer_9_4d, answer_9_4e, answer_9_4f]):
    wb.check(f"9.4{letter}", answer)

**Ex 9.5 — Early stopping avec patience sur une courbe de loss**

In [ ]:
# Ex 9.5: the values you found on paper
answer_9_5a = ...  # a) a whole number (an epoch)
answer_9_5b = ...  # b) a whole number (an epoch)
answer_9_5c = ...  # c) a whole number (an epoch)
answer_9_5d = ...  # d) a whole number (an epoch)
answer_9_5e = ...  # e) a whole number (an epoch)
answer_9_5f = ...  # f) a whole number (an epoch)
answer_9_5g = ...  # g) a whole number (an epoch)

for letter, answer in zip("abcdefg", [answer_9_5a, answer_9_5b, answer_9_5c, answer_9_5d, answer_9_5e, answer_9_5f, answer_9_5g]):
    wb.check(f"9.5{letter}", answer)

**Ex 9.6 — Lasso en dimension 1 : le seuillage doux et les zéros exacts**

In [ ]:
# Ex 9.6: the values you found on paper
answer_9_6a = ...  # a) a number
answer_9_6b = ...  # b) a number
answer_9_6c = ...  # c) a number
answer_9_6d = ...  # d) a number
answer_9_6e = ...  # e) a number
answer_9_6f = ...  # f) a number
answer_9_6g = ...  # g) a number
answer_9_6h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_9_6a, answer_9_6b, answer_9_6c, answer_9_6d, answer_9_6e, answer_9_6f, answer_9_6g, answer_9_6h]):
    wb.check(f"9.6{letter}", answer)

**Ex 9.7 — Mise à jour bayésienne d'une droite sur une grille 3 × 3**

In [ ]:
# Ex 9.7: the values you found on paper
answer_9_7a = ...  # a) a number
answer_9_7b = ...  # b) a number (3 decimals)
answer_9_7c = ...  # c) a whole number
answer_9_7d = ...  # d) a number (3 decimals)
answer_9_7e = ...  # e) a number (3 decimals)
answer_9_7f = ...  # f) True or False
answer_9_7g = ...  # g) a list [a, b]

for letter, answer in zip("abcdefg", [answer_9_7a, answer_9_7b, answer_9_7c, answer_9_7d, answer_9_7e, answer_9_7f, answer_9_7g]):
    wb.check(f"9.7{letter}", answer)

**Ex 9.9 — Diagnostiquer quatre paires de courbes d'entraînement et de validation**

In [ ]:
# Ex 9.9: the values you found on paper
answer_9_9a = ...  # a) a panel number (1 to 4)
answer_9_9b = ...  # b) a panel number (1 to 4)
answer_9_9c = ...  # c) one of 10, 30, 60 and 100
answer_9_9d = ...  # d) the letters, in alphabetical order, e.g. "BD"
answer_9_9e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_9_9a, answer_9_9b, answer_9_9c, answer_9_9d, answer_9_9e]):
    wb.check(f"9.9{letter}", answer)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu dire, à partir de deux erreurs et d'une référence, si un modèle sous-apprend ou sur-apprend, et quoi faire dans chaque cas ?
2. Sais-tu dériver les moindres carrés et Ridge en dimension 1, et expliquer pourquoi le Lasso met des poids exactement à zéro ?
3. Sais-tu calculer le biais² et la variance d'une famille de modèles, et dire pourquoi le compromis biais-variance n'est pas une loi ?

**Pour aller plus loin** : le guide « Linear Models » de scikit-learn et le chapitre 6 d'*An Introduction to Statistical Learning*, cités dans la fiche. La suite : le ch. 10 (neurones), où un modèle linéaire suivi d'une activation devient un neurone.

*Les exercices de code 9.12 à 9.31 (parties A à D, ta librairie `mylearn.linear`) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py 9` pour obtenir le notebook complet.*